# Step 4: Formal evaluation -- CVaR, drawdown, and statistical significance

We evaluate on the TEST set primarily -- this is the genuine out-of-sample comparison. Val is shown alongside for reference, but the test set is what goes in the paper's headline results.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

test = pd.read_csv("test_results.csv")
val = pd.read_csv("val_results.csv")
test["sample_date"] = pd.to_datetime(test["sample_date"])
val["sample_date"] = pd.to_datetime(val["sample_date"])
print(f"Test: {len(test)} rows ({test['symbol'].nunique()} unique symbols, {test['sample_date'].nunique()} sampled days)")
test.head()

## Metric 1-4: Mean, std, CVaR (95%), and a Sharpe-like ratio, per strategy

In [ ]:
def cvar_95(pnl_series):
    """Average P&L in the worst 5% of episodes (expected shortfall on the LEFT tail)."""
    threshold = pnl_series.quantile(0.05)
    return pnl_series[pnl_series <= threshold].mean()

def summarize(df, label):
    rows = []
    for strategy, group in df.groupby("strategy"):
        pnl = group["terminal_pnl"]
        rows.append({
            "strategy": strategy,
            "mean_pnl": pnl.mean(),
            "std_pnl": pnl.std(),
            "cvar_95": cvar_95(pnl),
            "sharpe_like": pnl.mean() / pnl.std() if pnl.std() > 0 else np.nan,
            "mean_cost": group["total_cost"].mean(),
            "mean_turnover": group["turnover"].mean(),
            "mean_n_trades": group["n_trades"].mean(),
        })
    result = pd.DataFrame(rows).set_index("strategy")
    print(f"=== {label} ===")
    print(result.round(4))
    print()
    return result

test_summary = summarize(test, "TEST set")
val_summary = summarize(val, "VAL set (for reference)")

## Metric 5: Max drawdown

Episodes are independent (different options, different days), so there's no single natural "path." To get a drawdown number, we chain episodes chronologically -- imagine running this strategy on one new option every sampled day, banking the terminal P&L each time -- and compute the running cumulative wealth and its max drawdown. This is a reasonable, explainable construction, worth stating explicitly in the paper's methodology.

In [ ]:
def max_drawdown(cumulative_series):
    running_max = cumulative_series.cummax()
    drawdown = cumulative_series - running_max
    return drawdown.min()

fig, ax = plt.subplots(figsize=(12, 5))
drawdowns = {}
for strategy, group in test.groupby("strategy"):
    ordered = group.sort_values("sample_date")
    # average terminal_pnl per day first (many episodes share a sample_date), then cumsum across days
    daily_avg = ordered.groupby("sample_date")["terminal_pnl"].mean()
    cumulative = daily_avg.cumsum()
    drawdowns[strategy] = max_drawdown(cumulative)
    ax.plot(cumulative.index, cumulative.values, marker="o", label=strategy)

ax.legend()
ax.set_title("Cumulative average P&L across sampled days (test set)")
plt.tight_layout()
plt.show()

print("Max drawdown by strategy:")
for k, v in drawdowns.items():
    print(f"  {k}: {v:.2f}")

## Statistical significance: block bootstrap

Episodes on the SAME sampled day share the same underlying price path, so they're correlated -- we can't treat them as independent. We resample whole DAYS (with replacement), not individual episodes, to preserve that within-day correlation. This gives us a confidence interval on the mean difference in terminal P&L between two strategies.

In [ ]:
def block_bootstrap_diff(df, strategy_a, strategy_b, n_boot=5000, seed=42):
    """Block bootstrap on the mean difference in terminal_pnl between two strategies, blocking by sample_date."""
    rng = np.random.default_rng(seed)
    unique_days = df["sample_date"].unique()

    a = df[df["strategy"] == strategy_a].set_index("sample_date")
    b = df[df["strategy"] == strategy_b].set_index("sample_date")

    # observed statistic
    observed_diff = a["terminal_pnl"].mean() - b["terminal_pnl"].mean()

    boot_diffs = np.zeros(n_boot)
    for i in range(n_boot):
        sampled_days = rng.choice(unique_days, size=len(unique_days), replace=True)
        a_vals = np.concatenate([a.loc[[d], "terminal_pnl"].values if d in a.index else [] for d in sampled_days])
        b_vals = np.concatenate([b.loc[[d], "terminal_pnl"].values if d in b.index else [] for d in sampled_days])
        boot_diffs[i] = a_vals.mean() - b_vals.mean()

    ci_lower, ci_upper = np.percentile(boot_diffs, [2.5, 97.5])
    # two-sided p-value: proportion of bootstrap draws on the opposite side of zero from the observed effect
    p_value = 2 * min((boot_diffs < 0).mean(), (boot_diffs > 0).mean())

    return observed_diff, ci_lower, ci_upper, p_value

print("Block bootstrap: Whalley-Wilmott vs BS delta (terminal P&L difference)")
diff, lo, hi, p = block_bootstrap_diff(test, "whalley_wilmott", "bs_delta")
print(f"  Observed mean difference: {diff:.4f}")
print(f"  95% CI: [{lo:.4f}, {hi:.4f}]")
print(f"  Approx two-sided p-value: {p:.4f}")

print("\nBlock bootstrap: Whalley-Wilmott vs Leland (terminal P&L difference)")
diff2, lo2, hi2, p2 = block_bootstrap_diff(test, "whalley_wilmott", "leland")
print(f"  Observed mean difference: {diff2:.4f}")
print(f"  95% CI: [{lo2:.4f}, {hi2:.4f}]")
print(f"  Approx two-sided p-value: {p2:.4f}")

## Also test the COST difference (not just P&L) -- this is where we expect the strongest, most reliable result

In [ ]:
def block_bootstrap_diff_col(df, strategy_a, strategy_b, col, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)
    unique_days = df["sample_date"].unique()
    a = df[df["strategy"] == strategy_a].set_index("sample_date")
    b = df[df["strategy"] == strategy_b].set_index("sample_date")
    observed_diff = a[col].mean() - b[col].mean()
    boot_diffs = np.zeros(n_boot)
    for i in range(n_boot):
        sampled_days = rng.choice(unique_days, size=len(unique_days), replace=True)
        a_vals = np.concatenate([a.loc[[d], col].values if d in a.index else [] for d in sampled_days])
        b_vals = np.concatenate([b.loc[[d], col].values if d in b.index else [] for d in sampled_days])
        boot_diffs[i] = a_vals.mean() - b_vals.mean()
    ci_lower, ci_upper = np.percentile(boot_diffs, [2.5, 97.5])
    p_value = 2 * min((boot_diffs < 0).mean(), (boot_diffs > 0).mean())
    return observed_diff, ci_lower, ci_upper, p_value

print("Block bootstrap: Whalley-Wilmott vs BS delta (COST difference)")
diff3, lo3, hi3, p3 = block_bootstrap_diff_col(test, "whalley_wilmott", "bs_delta", "total_cost")
print(f"  Observed mean cost difference: {diff3:.4f}")
print(f"  95% CI: [{lo3:.4f}, {hi3:.4f}]")
print(f"  Approx two-sided p-value: {p3:.4f}")